# A100 Long-Context Retrieval Benchmark (Dense vs Window vs Sparse)

This notebook is designed for **Colab A100 (High RAM)** and runs a publish-grade retrieval benchmark focused on long-context behavior.

It measures:
- retrieval quality (overall and distance-bucket accuracy)
- train throughput (`tok/s`)
- peak VRAM
- sparse diagnostics (`valid_neighbors`, entropy, window/expander mass when available)

Default setup compares:
- `dense`
- `window_w64`
- `sparse_w64_d8` (flex)
- `sparse_w64_d64_qknorm` (flex + qk norm)

All variants run with matched model size/optimizer/token budget for apples-to-apples comparison.


In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True
REPO_URL = "https://github.com/akashkguw/orion.git"
REPO_REF = os.environ.get("ORION_GIT_REF", "main")

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/orion_long_context")
    REPO_ROOT = DRIVE_ROOT / "repo"
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
else:
    REPO_ROOT = Path("/content/orion") if IN_COLAB else Path.cwd()


def _run(cmd: list[str], *, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed ({' '.join(cmd)}):\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    return proc


def _is_git_repo(path: Path) -> bool:
    return (path / ".git").exists()


def _clone_repo(repo_url: str, repo_root: Path, retries: int = 3) -> None:
    last_error = "unknown"
    for attempt in range(1, retries + 1):
        proc = subprocess.run(["git", "clone", repo_url, str(repo_root)], text=True, capture_output=True)
        if proc.returncode == 0:
            return

        last_error = (proc.stderr or proc.stdout or "clone failed").strip()
        tail = last_error.splitlines()[-1] if last_error else "clone failed"
        print(f"Clone attempt {attempt}/{retries} failed: {tail}")

        if repo_root.exists() and not _is_git_repo(repo_root):
            shutil.rmtree(repo_root, ignore_errors=True)

        if attempt < retries:
            time.sleep(attempt)

    raise RuntimeError(f"Failed to clone repository after {retries} attempts: {last_error}")


def _sync_repo(repo_root: Path, ref: str) -> None:
    status = _run(["git", "status", "--porcelain"], cwd=repo_root)
    if status.stdout.strip():
        raise RuntimeError(
            "Repository has local changes in Drive clone. Commit/stash/remove them, or delete the repo folder and rerun."
        )

    _run(["git", "fetch", "--all", "--tags", "--prune"], cwd=repo_root)

    checkout = _run(["git", "checkout", ref], cwd=repo_root, check=False)
    if checkout.returncode != 0:
        _run(["git", "checkout", "-B", ref, f"origin/{ref}"], cwd=repo_root)

    branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=repo_root).stdout.strip()
    if branch != "HEAD":
        _run(["git", "pull", "--ff-only", "origin", branch], cwd=repo_root)


if REPO_ROOT.exists() and not _is_git_repo(REPO_ROOT):
    if any(REPO_ROOT.iterdir()):
        backup = REPO_ROOT.with_name(f"{REPO_ROOT.name}_backup_{int(time.time())}")
        print(f"Found non-git directory at {REPO_ROOT}; moving to {backup}")
        REPO_ROOT.rename(backup)
    else:
        REPO_ROOT.rmdir()

if not REPO_ROOT.exists():
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print(f"Cloning {REPO_URL} -> {REPO_ROOT}")
    _clone_repo(REPO_URL, REPO_ROOT)
else:
    print(f"Using existing repository at {REPO_ROOT}")

_sync_repo(REPO_ROOT, REPO_REF)
os.chdir(REPO_ROOT)

commit = _run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT).stdout.strip()
branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=REPO_ROOT).stdout.strip()

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")
print(f"Checked out: {branch} @ {commit}")
if IN_COLAB and USE_GOOGLE_DRIVE:
    print(f"Google Drive persistence enabled at: {REPO_ROOT}")


In [ ]:
print(f"Python: {sys.version}")
if (sys.version_info.major, sys.version_info.minor) != (3, 11):
    print("Warning: Orion is tested primarily on Python 3.11. Continuing best-effort.")


def _pip_install(args: list[str], *, optional: bool = False) -> bool:
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.returncode == 0:
        return True

    print(f"\n[pip failed] {' '.join(cmd)}")
    if proc.stdout.strip():
        print("--- pip stdout (tail) ---")
        print("\n".join(proc.stdout.splitlines()[-40:]))
    if proc.stderr.strip():
        print("--- pip stderr (tail) ---")
        print("\n".join(proc.stderr.splitlines()[-80:]))

    if optional:
        print("Continuing because this dependency set is optional.")
        return False

    raise RuntimeError("Required dependency installation failed.")


_pip_install(["-U", "pip"])
_pip_install(["-r", "requirements.txt"])
_pip_install(["-e", "."])
_pip_install(["pandas", "seaborn", "matplotlib"])

print("Dependencies installed.")


In [ ]:
import gc
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.optim import AdamW

from orion.attention.base import AttentionConfig
from orion.models_factory import OrionDecoder
from orion.stability import StabilityConfig

sns.set_theme(style="whitegrid")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    print("cuda:", torch.version.cuda)

# ----------------------
# Experiment presets
# ----------------------
MODE = "a100_full"  # "quick" | "a100_full"
RUN_NAME = "experiment_a100_long_context_retrieval"
RESULTS_ROOT = Path("runs") / RUN_NAME
TRIAL_DIR = RESULTS_ROOT / "trials"
RESULTS_CSV = RESULTS_ROOT / "summary.csv"
RESUME = True
OVERWRITE = False

MODE_CFG = {
    "quick": {
        "seq_lens": [2048, 4096],
        "seeds": [123],
        "tokens_target": 1_500_000,
        "eval_batches": 24,
        "batch_by_seq": {2048: 4, 4096: 2},
    },
    "a100_full": {
        "seq_lens": [2048, 4096, 8192],
        "seeds": [123, 456, 789],
        "tokens_target": 4_000_000,
        "eval_batches": 64,
        "batch_by_seq": {2048: 8, 4096: 4, 8192: 1},
    },
}

if MODE not in MODE_CFG:
    raise ValueError(f"Unknown MODE={MODE!r}")

cfg = MODE_CFG[MODE]
SEQ_LENS = cfg["seq_lens"]
SEEDS = cfg["seeds"]
TOKENS_TARGET = int(cfg["tokens_target"])
EVAL_BATCHES = int(cfg["eval_batches"])
BATCH_BY_SEQ = dict(cfg["batch_by_seq"])

# ----------------------
# Model + task settings
# ----------------------
DMODEL = 256
NLAYERS = 4
NHEADS = 4
MLP_MULT = 4
LR = 3e-4
WEIGHT_DECAY = 0.01
LOG_EVERY = 25

VOCAB_SIZE = 512
KEY_VOCAB = 192
VALUE_VOCAB = 192
VALUE_BASE = KEY_VOCAB + 1
QUERY_TOKEN = VOCAB_SIZE - 1
FILLER_LOW = VALUE_BASE + VALUE_VOCAB
FILLER_HIGH = VOCAB_SIZE - 2
NUM_PAIRS = 96

if not (2 * NUM_PAIRS + 3 <= min(SEQ_LENS)):
    raise ValueError("NUM_PAIRS is too large for the chosen minimum seq_len.")
if FILLER_LOW > FILLER_HIGH:
    raise ValueError("Invalid filler token range. Increase VOCAB_SIZE.")

VARIANTS = [
    {"variant_id": "dense", "backend": "dense", "window_size": None, "expander_degree": None, "qk_norm": False},
    {"variant_id": "window_w64", "backend": "window", "window_size": 64, "expander_degree": None, "qk_norm": False},
    {
        "variant_id": "sparse_w64_d8",
        "backend": "sparse",
        "window_size": 64,
        "expander_degree": 8,
        "qk_norm": False,
        "sparse_impl": "flex",
        "sparse_block_size": 128,
    },
    {
        "variant_id": "sparse_w64_d64_qknorm",
        "backend": "sparse",
        "window_size": 64,
        "expander_degree": 64,
        "qk_norm": True,
        "sparse_impl": "flex",
        "sparse_block_size": 128,
    },
]

for seq_len in SEQ_LENS:
    if seq_len % 64 != 0:
        raise ValueError(f"seq_len={seq_len} must be a multiple of 64 for stable flex kernels.")

for v in VARIANTS:
    if v["backend"] == "sparse":
        bs = int(v.get("sparse_block_size", 128))
        if bs % 64 != 0:
            raise ValueError(f"{v['variant_id']}: sparse_block_size must be a multiple of 64.")

TRIALS = []
for variant in VARIANTS:
    for seq_len in SEQ_LENS:
        for seed in SEEDS:
            if seq_len not in BATCH_BY_SEQ:
                raise KeyError(f"Missing batch size for seq_len={seq_len}")
            TRIALS.append(
                {
                    **variant,
                    "seq_len": int(seq_len),
                    "seed": int(seed),
                    "batch_size": int(BATCH_BY_SEQ[seq_len]),
                    "steps": max(1, math.ceil(TOKENS_TARGET / (BATCH_BY_SEQ[seq_len] * seq_len))),
                }
            )

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
TRIAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODE={MODE} | trials={len(TRIALS)}")
print(f"results: {RESULTS_ROOT}")
display(pd.DataFrame(TRIALS).head(20))


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_attention_cfg(variant: dict) -> AttentionConfig:
    backend = variant["backend"]
    return AttentionConfig(
        backend=backend,
        window_size=variant.get("window_size"),
        expander_degree=variant.get("expander_degree"),
        sparse_impl=variant.get("sparse_impl", "flex"),
        sparse_block_size=int(variant.get("sparse_block_size", 128)),
        sparse_probe_every=0,
        sparse_probe_tokens=256,
        window_probe_every=0,
        window_probe_tokens=256,
    )


def build_model_for_trial(variant: dict, seq_len: int) -> torch.nn.Module:
    attention_cfg = build_attention_cfg(variant)
    stability_cfg = None
    if variant["backend"] == "sparse":
        stability_cfg = StabilityConfig(
            qk_norm=bool(variant.get("qk_norm", False)),
            ortho_init=False,
            spectral_norm=False,
        )

    model = OrionDecoder(
        vocab_size=VOCAB_SIZE,
        d_model=DMODEL,
        n_layers=NLAYERS,
        n_heads=NHEADS,
        mlp_mult=MLP_MULT,
        attention_cfg=attention_cfg,
        max_seq_len=seq_len,
        stability_cfg=stability_cfg,
    ).to(DEVICE)
    return model


def _unique_keys(batch_size: int, num_pairs: int, *, generator: torch.Generator) -> torch.Tensor:
    keys = torch.empty((batch_size, num_pairs), dtype=torch.long, device=DEVICE)
    for b in range(batch_size):
        perm = torch.randperm(KEY_VOCAB, device=DEVICE, generator=generator)
        keys[b] = perm[:num_pairs] + 1
    return keys


def make_retrieval_batch(
    batch_size: int,
    seq_len: int,
    *,
    generator: torch.Generator,
) -> tuple[torch.Tensor, torch.Tensor, int, torch.Tensor]:
    # Sequence template:
    # [k1, v1, k2, v2, ..., filler..., QUERY, kq, vq]
    seq = torch.randint(
        FILLER_LOW,
        FILLER_HIGH + 1,
        (batch_size, seq_len),
        device=DEVICE,
        generator=generator,
    )

    keys = _unique_keys(batch_size, NUM_PAIRS, generator=generator)
    values = torch.randint(
        VALUE_BASE,
        VALUE_BASE + VALUE_VOCAB,
        (batch_size, NUM_PAIRS),
        device=DEVICE,
        generator=generator,
    )

    pair_pos = torch.arange(0, 2 * NUM_PAIRS, 2, device=DEVICE)
    seq[:, pair_pos] = keys
    seq[:, pair_pos + 1] = values

    query_pair_idx = torch.randint(0, NUM_PAIRS, (batch_size,), device=DEVICE, generator=generator)

    q_marker_pos = seq_len - 3
    q_key_pos = seq_len - 2
    q_value_pos = seq_len - 1

    seq[:, q_marker_pos] = QUERY_TOKEN
    row = torch.arange(batch_size, device=DEVICE)
    seq[:, q_key_pos] = keys[row, query_pair_idx]
    seq[:, q_value_pos] = values[row, query_pair_idx]

    mem_key_pos = 2 * query_pair_idx
    distances = (q_key_pos - mem_key_pos).to(torch.long)

    x = seq[:, :-1]
    y = seq[:, 1:]

    return x, y, q_key_pos, distances


def _find_first_attention_backend(model: torch.nn.Module):
    for module in model.modules():
        backend = getattr(module, "attn", None)
        if backend is not None:
            return backend
    return None


def _distance_bucket(dist: int) -> str:
    if dist <= 512:
        return "d_le_512"
    if dist <= 1024:
        return "d_513_1024"
    if dist <= 2048:
        return "d_1025_2048"
    if dist <= 4096:
        return "d_2049_4096"
    return "d_gt_4096"


def evaluate_retrieval(
    model: torch.nn.Module,
    *,
    seq_len: int,
    batch_size: int,
    eval_batches: int,
    seed: int,
) -> dict:
    model.eval()
    gen_device = DEVICE if DEVICE.type == "cuda" else torch.device("cpu")
    g = torch.Generator(device=gen_device)
    g.manual_seed(seed + 10_000)

    correct = 0
    total = 0
    bucket_hit = {}
    bucket_tot = {}

    use_amp = DEVICE.type == "cuda"
    amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16

    with torch.no_grad():
        for _ in range(eval_batches):
            x, y, q_pos, distances = make_retrieval_batch(batch_size, seq_len, generator=g)
            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
                logits = model(x)
                q_logits = logits[:, q_pos, :]
            q_targets = y[:, q_pos]
            preds = q_logits.argmax(dim=-1)
            hits = (preds == q_targets)

            correct += int(hits.sum().item())
            total += int(hits.numel())

            dist_cpu = distances.detach().cpu().tolist()
            hit_cpu = hits.detach().cpu().tolist()
            for d, h in zip(dist_cpu, hit_cpu):
                b = _distance_bucket(int(d))
                bucket_tot[b] = bucket_tot.get(b, 0) + 1
                bucket_hit[b] = bucket_hit.get(b, 0) + int(h)

    bucket_acc = {
        f"acc_{k}": (bucket_hit[k] / bucket_tot[k]) if bucket_tot.get(k, 0) > 0 else np.nan
        for k in ["d_le_512", "d_513_1024", "d_1025_2048", "d_2049_4096", "d_gt_4096"]
    }

    return {
        "retrieval_acc": (correct / total) if total > 0 else np.nan,
        **bucket_acc,
    }


In [ ]:
def run_trial(trial: dict) -> dict:
    trial_id = f"{trial['variant_id']}_T{trial['seq_len']}_seed{trial['seed']}"
    print(f"\n>>> {trial_id} | bs={trial['batch_size']} steps={trial['steps']}")

    set_seed(int(trial["seed"]))
    model = None

    try:
        model = build_model_for_trial(trial, trial["seq_len"])
        optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        use_amp = DEVICE.type == "cuda"
        amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
        scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and amp_dtype == torch.float16))

        gen_device = DEVICE if DEVICE.type == "cuda" else torch.device("cpu")
        train_gen = torch.Generator(device=gen_device)
        train_gen.manual_seed(int(trial["seed"]))

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        model.train()
        t0 = time.time()
        losses = []
        accs = []

        for step in range(1, int(trial["steps"]) + 1):
            x, y, q_pos, _dist = make_retrieval_batch(
                int(trial["batch_size"]),
                int(trial["seq_len"]),
                generator=train_gen,
            )

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
                logits = model(x)
                q_logits = logits[:, q_pos, :]
                q_targets = y[:, q_pos]
                loss = F.cross_entropy(q_logits, q_targets)

            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            with torch.no_grad():
                preds = q_logits.argmax(dim=-1)
                acc = (preds == q_targets).float().mean().item()

            losses.append(float(loss.item()))
            accs.append(float(acc))

            if step % LOG_EVERY == 0 or step == 1 or step == int(trial["steps"]):
                print(f"step {step:4d}: loss={losses[-1]:.4f} acc={accs[-1]:.4f}")

        train_dt = time.time() - t0
        train_tokens = int(trial["steps"]) * int(trial["batch_size"]) * int(trial["seq_len"])
        train_tok_s = train_tokens / train_dt if train_dt > 0 else np.nan
        step_time_ms = 1000.0 * train_dt / max(1, int(trial["steps"]))

        peak_vram_gb = np.nan
        if DEVICE.type == "cuda":
            peak_vram_gb = float(torch.cuda.max_memory_allocated() / (1024**3))

        eval_stats = evaluate_retrieval(
            model,
            seq_len=int(trial["seq_len"]),
            batch_size=max(4, int(trial["batch_size"])),
            eval_batches=EVAL_BATCHES,
            seed=int(trial["seed"]),
        )

        backend_obj = _find_first_attention_backend(model)
        attn_entropy_norm = float(getattr(backend_obj, "last_attn_entropy_normalized", np.nan))
        valid_neighbors = float(getattr(backend_obj, "last_valid_neighbor_fraction", np.nan))
        valid_vs_cap = float(
            getattr(backend_obj, "last_valid_neighbor_fraction_vs_causal_cap", np.nan)
        )
        window_mass = float(getattr(backend_obj, "last_attention_mass_window_pct", np.nan))
        expander_mass = float(getattr(backend_obj, "last_attention_mass_expander_pct", np.nan))

        row = {
            "trial_id": trial_id,
            "status": "ok",
            "backend": trial["backend"],
            "variant_id": trial["variant_id"],
            "seq_len": int(trial["seq_len"]),
            "seed": int(trial["seed"]),
            "batch_size": int(trial["batch_size"]),
            "steps": int(trial["steps"]),
            "train_tokens": int(train_tokens),
            "train_seconds": float(train_dt),
            "train_tok_per_s": float(train_tok_s),
            "step_time_ms": float(step_time_ms),
            "peak_vram_gb": float(peak_vram_gb),
            "final_loss": float(np.mean(losses[-25:])),
            "final_acc": float(np.mean(accs[-25:])),
            "attn_entropy_norm": attn_entropy_norm,
            "valid_neighbors": valid_neighbors,
            "valid_vs_causal_cap": valid_vs_cap,
            "window_mass_pct": window_mass,
            "expander_mass_pct": expander_mass,
            **eval_stats,
            "error": "",
        }

    except Exception as exc:
        row = {
            "trial_id": trial_id,
            "status": "failed",
            "backend": trial["backend"],
            "variant_id": trial["variant_id"],
            "seq_len": int(trial["seq_len"]),
            "seed": int(trial["seed"]),
            "batch_size": int(trial["batch_size"]),
            "steps": int(trial["steps"]),
            "error": f"{type(exc).__name__}: {exc}",
        }
        print("[failed]", row["error"])

    finally:
        if model is not None:
            del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    trial_json = TRIAL_DIR / f"{trial_id}.json"
    trial_json.write_text(json.dumps(row, indent=2), encoding="utf-8")
    return row


In [ ]:
existing = pd.DataFrame()
if RESUME and RESULTS_CSV.exists() and not OVERWRITE:
    existing = pd.read_csv(RESULTS_CSV)
    print(f"Loaded existing results: {len(existing)} rows")


def _done_key(df: pd.DataFrame) -> set[tuple[str, int, int]]:
    if df.empty:
        return set()
    keep = df[df["status"] == "ok"]
    return set(
        zip(
            keep["variant_id"].astype(str),
            keep["seq_len"].astype(int),
            keep["seed"].astype(int),
        )
    )


done = _done_key(existing)
rows = [] if existing.empty or OVERWRITE else existing.to_dict("records")

for idx, trial in enumerate(TRIALS, start=1):
    key = (trial["variant_id"], int(trial["seq_len"]), int(trial["seed"]))
    if RESUME and (not OVERWRITE) and key in done:
        print(f"[{idx}/{len(TRIALS)}] skip {trial['variant_id']} T{trial['seq_len']} seed{trial['seed']}")
        continue

    print(f"[{idx}/{len(TRIALS)}] run {trial['variant_id']} T{trial['seq_len']} seed{trial['seed']}")
    row = run_trial(trial)
    rows = [r for r in rows if not (
        str(r.get("variant_id")) == trial["variant_id"]
        and int(r.get("seq_len", -1)) == int(trial["seq_len"])
        and int(r.get("seed", -1)) == int(trial["seed"])
    )]
    rows.append(row)

    df_now = pd.DataFrame(rows)
    df_now.sort_values(["variant_id", "seq_len", "seed"], inplace=True, ignore_index=True)
    df_now.to_csv(RESULTS_CSV, index=False)
    print(f"Saved: {RESULTS_CSV} ({len(df_now)} rows)")

results = pd.read_csv(RESULTS_CSV)
print("\nRun status counts:")
display(results["status"].value_counts(dropna=False).to_frame("count"))
print("Results path:", RESULTS_CSV)


In [ ]:
results = pd.read_csv(RESULTS_CSV)
df_ok = results[results["status"] == "ok"].copy()

if df_ok.empty:
    raise ValueError("No successful trials found.")

for col in [
    "train_tok_per_s",
    "step_time_ms",
    "peak_vram_gb",
    "retrieval_acc",
    "final_loss",
    "final_acc",
]:
    if col in df_ok.columns:
        df_ok[col] = pd.to_numeric(df_ok[col], errors="coerce")

summary = (
    df_ok.groupby(["backend", "variant_id", "seq_len"], as_index=False)
    .agg(
        seeds=("seed", "nunique"),
        retrieval_acc_mean=("retrieval_acc", "mean"),
        retrieval_acc_std=("retrieval_acc", "std"),
        train_tok_s_mean=("train_tok_per_s", "mean"),
        train_tok_s_std=("train_tok_per_s", "std"),
        peak_vram_gb_mean=("peak_vram_gb", "mean"),
        peak_vram_gb_std=("peak_vram_gb", "std"),
        final_loss_mean=("final_loss", "mean"),
        final_loss_std=("final_loss", "std"),
    )
    .sort_values(["seq_len", "backend", "variant_id"])
)

print("Mean ± std summary:")
display(summary)

pair_keys = ["seq_len", "seed"]
dense = df_ok[df_ok["backend"] == "dense"][pair_keys + ["train_tok_per_s", "peak_vram_gb", "retrieval_acc"]].rename(
    columns={
        "train_tok_per_s": "dense_train_tok_s",
        "peak_vram_gb": "dense_peak_vram_gb",
        "retrieval_acc": "dense_retrieval_acc",
    }
)
non_dense = df_ok[df_ok["backend"] != "dense"].copy()
paired = non_dense.merge(dense, on=pair_keys, how="inner")
paired["speed_ratio_vs_dense"] = paired["train_tok_per_s"] / paired["dense_train_tok_s"]
paired["vram_ratio_vs_dense"] = paired["peak_vram_gb"] / paired["dense_peak_vram_gb"]
paired["retrieval_delta_vs_dense"] = paired["retrieval_acc"] - paired["dense_retrieval_acc"]

paired_summary = (
    paired.groupby(["backend", "variant_id", "seq_len"], as_index=False)
    .agg(
        runs=("seed", "count"),
        speed_ratio_mean=("speed_ratio_vs_dense", "mean"),
        speed_ratio_std=("speed_ratio_vs_dense", "std"),
        vram_ratio_mean=("vram_ratio_vs_dense", "mean"),
        vram_ratio_std=("vram_ratio_vs_dense", "std"),
        retrieval_delta_mean=("retrieval_delta_vs_dense", "mean"),
        retrieval_delta_std=("retrieval_delta_vs_dense", "std"),
    )
    .sort_values(["seq_len", "backend", "variant_id"])
)

print("Paired summary vs dense:")
display(paired_summary)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.lineplot(
    data=df_ok,
    x="seq_len",
    y="train_tok_per_s",
    hue="variant_id",
    marker="o",
    estimator="mean",
    errorbar="sd",
    ax=axes[0],
)
axes[0].set_title("Train Throughput vs Seq Len")
axes[0].set_ylabel("tokens/sec")

sns.lineplot(
    data=df_ok,
    x="seq_len",
    y="peak_vram_gb",
    hue="variant_id",
    marker="o",
    estimator="mean",
    errorbar="sd",
    ax=axes[1],
)
axes[1].set_title("Peak VRAM vs Seq Len")
axes[1].set_ylabel("GB")

sns.lineplot(
    data=df_ok,
    x="seq_len",
    y="retrieval_acc",
    hue="variant_id",
    marker="o",
    estimator="mean",
    errorbar="sd",
    ax=axes[2],
)
axes[2].set_title("Retrieval Accuracy vs Seq Len")
axes[2].set_ylabel("accuracy")
axes[2].set_ylim(0.0, 1.05)

plt.tight_layout()
plt.show()


In [ ]:
bucket_cols = ["acc_d_le_512", "acc_d_513_1024", "acc_d_1025_2048", "acc_d_2049_4096", "acc_d_gt_4096"]

avail = [c for c in bucket_cols if c in df_ok.columns]
if avail:
    bucket_df = (
        df_ok.groupby(["variant_id", "seq_len"], as_index=False)[avail]
        .mean()
        .sort_values(["variant_id", "seq_len"])
    )

    print("Distance-bucket retrieval accuracy (mean):")
    display(bucket_df)

    melt = bucket_df.melt(
        id_vars=["variant_id", "seq_len"],
        value_vars=avail,
        var_name="distance_bucket",
        value_name="accuracy",
    )

    for seq_len in sorted(melt["seq_len"].unique()):
        pivot = (
            melt[melt["seq_len"] == seq_len]
            .pivot(index="variant_id", columns="distance_bucket", values="accuracy")
            .reindex(columns=avail)
        )

        plt.figure(figsize=(8, 3.8))
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", vmin=0.0, vmax=1.0)
        plt.title(f"Distance-Bucket Accuracy @ seq_len={seq_len}")
        plt.xlabel("distance bucket")
        plt.ylabel("variant")
        plt.tight_layout()
        plt.show()
else:
    print("No distance-bucket columns found.")


## Notes

- This notebook is **resume-safe** via `runs/experiment_a100_long_context_retrieval/summary.csv`.
- To rerun from scratch, set `OVERWRITE = True`.
- To reduce runtime, switch `MODE` to `quick`.
- Sparse runs are configured with `sparse_impl='flex'` only.
